# Audio Recorder

This notebook records audio from the default macOS microphone and saves it as a WAV file.

## Troubleshooting Empty Recordings

If you're getting empty/silent recordings, check:

1. **Microphone Permissions**: macOS requires permission for apps to access the microphone
   - Go to: System Preferences → Security & Privacy → Privacy → Microphone
   - Ensure Terminal (or your IDE) has microphone access enabled
   - You may need to restart Jupyter after granting permission

2. **Correct Input Device**: Make sure the right microphone is selected
   - Run the device check cell below to see available devices
   - Check System Preferences → Sound → Input to verify the active microphone

3. **Microphone Not Muted**: Check that your microphone isn't muted in system settings

4. **Input Volume**: Ensure the input volume is turned up in System Preferences → Sound → Input

In [9]:
# Import required libraries
import sounddevice as sd
import numpy as np
from scipy.io import wavfile
from datetime import datetime
import os

In [10]:
# Configuration parameters
SAMPLE_RATE = 16000  # Hz
CHANNELS = 1  # Mono audio
DURATION = 6  # seconds (configurable)
DTYPE = np.int16  # 16-bit PCM format

In [ ]:
# Check available audio devices
print("Available audio devices:")
print(sd.query_devices())
print("\nDefault input device:")
print(sd.query_devices(kind='input'))

In [ ]:
def check_audio_data(audio_data, sample_rate=SAMPLE_RATE):
    """
    Check if recorded audio contains actual sound or is empty/silent.
    
    Parameters:
    -----------
    audio_data : numpy.ndarray
        Audio data to check
    sample_rate : int
        Sample rate in Hz
    
    Returns:
    --------
    bool
        True if audio contains sound, False if silent/empty
    """
    # Calculate statistics
    max_amplitude = np.max(np.abs(audio_data))
    rms = np.sqrt(np.mean(audio_data.astype(float)**2))
    
    print("\n=== Audio Quality Check ===")
    print(f"Duration: {len(audio_data) / sample_rate:.2f} seconds")
    print(f"Samples: {len(audio_data)}")
    print(f"Max amplitude: {max_amplitude} (out of 32767)")
    print(f"RMS level: {rms:.2f}")
    print(f"Peak percentage: {(max_amplitude / 32767) * 100:.2f}%")
    
    # Check if audio is essentially silent
    # Threshold: max amplitude should be at least 100 (about 0.3% of max)
    if max_amplitude < 100:
        print("\n⚠️  WARNING: Recording appears to be SILENT or EMPTY!")
        print("Possible issues:")
        print("  - Microphone not selected or muted")
        print("  - No microphone permission granted to Terminal/Jupyter")
        print("  - Wrong input device selected")
        print("  - Microphone hardware issue")
        return False
    elif max_amplitude < 1000:
        print("\n⚠️  WARNING: Recording level is VERY LOW")
        print("  - Check microphone volume/gain settings")
        print("  - Speak louder or move closer to microphone")
        return True
    else:
        print("\n✓ Recording looks good!")
        return True

In [11]:
def record_audio(duration=DURATION, sample_rate=SAMPLE_RATE, channels=CHANNELS):
    """
    Record audio from the default microphone.
    
    Parameters:
    -----------
    duration : int
        Recording duration in seconds (default: 30)
    sample_rate : int
        Sample rate in Hz (default: 16000)
    channels : int
        Number of channels - 1 for mono, 2 for stereo (default: 1)
    
    Returns:
    --------
    numpy.ndarray
        Recorded audio data
    """
    print(f"Recording started... ({duration} seconds)")
    print(f"Sample rate: {sample_rate} Hz, Channels: {channels}")
    print("🎤 Speak now!\n")
    
    # Record audio
    recording = sd.rec(
        int(duration * sample_rate),
        samplerate=sample_rate,
        channels=channels,
        dtype='float32'
    )
    
    # Wait until recording is finished
    sd.wait()
    
    print("Recording finished!")
    
    # Convert to 16-bit PCM format
    recording_int16 = np.int16(recording * 32767)
    
    # Check if recording contains audio
    check_audio_data(recording_int16, sample_rate)
    
    return recording_int16

In [12]:
def save_audio(audio_data, sample_rate=SAMPLE_RATE, output_dir='recordings', force_save=False):
    """
    Save audio data to a WAV file with timestamp in filename.
    
    Parameters:
    -----------
    audio_data : numpy.ndarray
        Audio data to save
    sample_rate : int
        Sample rate in Hz (default: 16000)
    output_dir : str
        Directory to save the file (default: 'recordings')
    force_save : bool
        If True, save even if audio appears silent (default: False)
    
    Returns:
    --------
    str or None
        Path to the saved file, or None if not saved
    """
    # Check if audio is silent (unless force_save is True)
    if not force_save:
        max_amplitude = np.max(np.abs(audio_data))
        if max_amplitude < 100:
            print("\n❌ NOT SAVING: Audio is silent/empty.")
            print("   Fix the microphone issue and try again.")
            print("   Or use save_audio(audio_data, force_save=True) to save anyway.")
            return None
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Generate filename with timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename = f"recording_{timestamp}.wav"
    filepath = os.path.join(output_dir, filename)
    
    # Save as WAV file
    wavfile.write(filepath, sample_rate, audio_data)
    
    print(f"\n✓ Audio saved to: {filepath}")
    return filepath

In [9]:
# Record and save audio
# Adjust DURATION variable above to change recording length
audio_data = record_audio(duration=DURATION)
filepath = save_audio(audio_data)

Recording started... (6 seconds)
Sample rate: 16000 Hz, Channels: 1
Recording finished!
Audio saved to: recordings/recording_20260111_195622.wav


In [ ]:
# Optional: Record with custom duration
# Uncomment and run this cell to record with a different duration
# custom_duration = 10  # seconds
# audio_data = record_audio(duration=custom_duration)
# filepath = save_audio(audio_data)